# 05 (4B) - Build training files for the 4B model
Same as notebook 05, using the 4B model's pairs and the 27B teacher's reviews of them.
Outputs: `sft_review_4b.json` (self-refine with review) and `sft_cot_4b.json` (conventional CoT on the same problems).

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

In [2]:
path1=Path('../datasets/review_pairs_qwen4b.json') # pairs from the 4B model (notebook 03_select_pairs_4b)
path2=Path('../datasets/reviews_qwen27b_4b.json') # 27B reviews of the 4B pairs (notebook 04b, 4B run)

with path1.open('r',encoding='utf-8') as file:
    pairs=json.load(file)

with path2.open('r',encoding='utf-8') as file:
    reviews=json.load(file)

print(len(pairs),len(reviews))

2677 2658


In [3]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [4]:
random.seed(0)

review_data=[]
refine_data=[]
cot_data=[]

for i in reviews:
    final_answer=extract_boxed(i['review'])

    # C: two chains + teacher review
    review_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': [i['answer_1'],i['answer_2']],
        'review': i['review'],
        'final_answer': final_answer,
    })

    # B: two chains + one more correct chain (prefer one the model has not shown yet)
    shown=[i['answer_1'],i['answer_2']]
    unseen=[c for c in i['other_correct'] if c not in shown]
    refined=random.choice(unseen) if unseen else random.choice(i['other_correct'])

    refine_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': shown,
        'refined': refined,
        'final_answer': extract_boxed(refined),
    })

    # conventional CoT: one correct chain, same problems
    cot_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': [random.choice(i['other_correct'])],
        'final_answer': None,
    })
    cot_data[-1]['final_answer']=extract_boxed(cot_data[-1]['chains'][0])

print(len(review_data),len(refine_data),len(cot_data))

2658 2658 2658


In [5]:
Counter(i['type'] for i in review_data)

Counter({'agree': 1000, 'correction': 829, 'keep': 829})

In [6]:
assert all(i['final_answer']!=None for i in review_data)
assert all(i['final_answer']!=None for i in refine_data)
assert all(i['final_answer']!=None for i in cot_data)
assert all(len(i['chains'])==2 for i in review_data)
assert all(len(i['chains'])==1 for i in cot_data)

In [7]:
review_data[0]

{'id': 'math-7401',
 'type': 'agree',
 'question': 'Given that $x$ is real and $x^3+\\frac{1}{x^3}=52$, find $x+\\frac{1}{x}$.',
 'chains': ['We are given that \\(x\\) is a real number satisfying \\[x^3+\\frac{1}{x^3}=52.\\] Let \\(y = x + \\frac{1}{x}\\). Then we can express \\(x^3+\\frac{1}{x^3}\\) in terms of \\(y\\). First, note that: \\[y = x + \\frac{1}{x} \\implies y^2 = x^2 + 2 + \\frac{1}{x^2} \\implies x^2 + \\frac{1}{x^2} = y^2 - 2.\\] Squaring both sides of the equation for \\(y^2\\), we get: \\[y^3 = x^3 + 3x + \\frac{3}{x} + \\frac{1}{x^3}.\\] This can be rewritten as: \\[y^3 = \\left(x^3 + \\frac{1}{x^3}\\right) + 3\\left(x + \\frac{1}{x}\\right).\\] Substituting the given value \\(x^3 + \\frac{1}{x^3} = 52\\) and \\(y = x + \\frac{1}{x}\\), we get: \\[y^3 = 52 + 3y.\\] Rearranging this equation, we have: \\[y^3 - 3y - 52 = 0.\\] Now, we need to solve the cubic equation \\(y^3 - 3y - 52 = 0\\). We can try to find rational roots using the Rational Root Theorem, which sugg

In [8]:
refine_data[0]

{'id': 'math-7401',
 'type': 'agree',
 'question': 'Given that $x$ is real and $x^3+\\frac{1}{x^3}=52$, find $x+\\frac{1}{x}$.',
 'chains': ['We are given that \\(x\\) is a real number satisfying \\[x^3+\\frac{1}{x^3}=52.\\] Let \\(y = x + \\frac{1}{x}\\). Then we can express \\(x^3+\\frac{1}{x^3}\\) in terms of \\(y\\). First, note that: \\[y = x + \\frac{1}{x} \\implies y^2 = x^2 + 2 + \\frac{1}{x^2} \\implies x^2 + \\frac{1}{x^2} = y^2 - 2.\\] Squaring both sides of the equation for \\(y^2\\), we get: \\[y^3 = x^3 + 3x + \\frac{3}{x} + \\frac{1}{x^3}.\\] This can be rewritten as: \\[y^3 = \\left(x^3 + \\frac{1}{x^3}\\right) + 3\\left(x + \\frac{1}{x}\\right).\\] Substituting the given value \\(x^3 + \\frac{1}{x^3} = 52\\) and \\(y = x + \\frac{1}{x}\\), we get: \\[y^3 = 52 + 3y.\\] Rearranging this equation, we have: \\[y^3 - 3y - 52 = 0.\\] Now, we need to solve the cubic equation \\(y^3 - 3y - 52 = 0\\). We can try to find rational roots using the Rational Root Theorem, which sugg

In [9]:
savepath1=Path('../datasets/sft_review_4b.json')
savepath2=Path('../datasets/sft_cot_4b.json')

with savepath1.open('w',encoding='utf-8') as file:
    json.dump(review_data,file,ensure_ascii=False,indent=2)

with savepath2.open('w',encoding='utf-8') as file:
    json.dump(cot_data,file,ensure_ascii=False,indent=2)

print("Saved:",savepath1,savepath2)

Saved: ../datasets/sft_review_4b.json ../datasets/sft_cot_4b.json
